In [ ]:
# Function to compare answers:
# - For non-numeric answers: exact match required (case-insensitive)
# - For numeric answers: considered correct if within 5% of expected value
def compare_answers(extracted, correct):
    # Try to convert to numeric values
    try:
        # Remove any '%' signs and convert to float
        extracted_num = float(extracted.replace('%', '').strip())
        correct_num = float(str(correct).replace('%', '').strip())
        
        # If both are numeric, apply 5% tolerance
        tolerance = 0.05  # 5%
        max_diff = correct_num * tolerance
        
        # Return True if within tolerance
        is_within_tolerance = abs(extracted_num - correct_num) <= max_diff
        print(f"Numeric comparison: {extracted_num} vs {correct_num}, difference: {abs(extracted_num - correct_num)}, max allowed: {max_diff}")
        return is_within_tolerance
        
    except (ValueError, TypeError):
        # If conversion fails, they're non-numeric, so require exact match (case-insensitive)
        exact_match = extracted.strip().lower() == str(correct).strip().lower()
        print(f"Text comparison (exact match required): {exact_match}")
        return exact_match
import requests
import base64
import os, time
import pandas as pd
import re

# Claude API Key
api_key = os.environ.get("ANTHROPIC_API_KEY", "")  # set ANTHROPIC_API_KEY
CLAUDE_API_URL = "https://api.anthropic.com/v1/messages"

# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

# Load questions dataset
questions_df = pd.read_csv("../Questions.csv")
# Process all questions in the dataset
questions = [question for _, question in questions_df.iterrows()]

# Function to extract answer from Claude's response
def extract_answer(response_text):
    # Use regex to find text after "Correct Answer: "
    match = re.search(r"Correct Answer:\s*([^\n]*)", response_text)
    if match:
        # Extract the answer and remove any leading/trailing spaces
        return match.group(1).strip()
    
    # If no match is found, look for the last statement in the text that might be an answer
    lines = response_text.strip().split('\n')
    non_empty_lines = [line.strip() for line in lines if line.strip()]
    
    if non_empty_lines:
        # Return the last non-empty line as a fallback
        last_line = non_empty_lines[-1]
        # If the last line is very long, it's probably not an answer
        if len(last_line) < 100:
            return last_line
    
    return ""

responses_data = []
for i, question in enumerate(questions, start=1):
  print(f"Starting iteration {i}...")  # Print at the start of each iteration
  
  question_text = question['question']
  question_answer = question["correct"]
  chart_name = question['vis']
  
  # Data
  data_files = set([f for f in os.listdir("../Data")])
  if chart_name + ".csv" in data_files:
    data_file = chart_name + ".csv"
  else:
    data_file = chart_name + ".json"
    
  with open('../Data/' + data_file, 'r', encoding='utf-8') as f:
      data_raw = f.read()
      
  # Path to your image
  image_path = "../Images/" + chart_name + ".png"
  
  # Getting the base64 string
  base64_image = encode_image(image_path)
  
  headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key,
    "anthropic-version": "2023-06-01"
  }
  
  headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key,
    "anthropic-version": "2023-06-01"
  }
  
  payload = {
    "model": "claude-3-7-sonnet-20250219",  # Using Claude 3.7 Sonnet model
    "max_tokens": 5000,
    "temperature": 0.0,
    "system": "You are an assistant, skilled in reading and interpreting visually represented data.",
    "messages": [
      {
        "role": "user",
        "content": 'I am about to show you a graph and ask you a multiple-choice question about that graph. \n\n' +
                   'Task 1: Data Extraction and Table Creation: First, explicitly list ALL numerical values you can identify on both axes, then create a structured table using markdown syntax that includes ALL data points you identified above with appropriate column headers with units. \n \n' +
                   'Task 2: Sort the data: Sort the data in descending order by the numerical values. \n \n' +
                   'Task 3: Data Verification and Error Handling: Double-check if your table matches ALL elements in the graph by comparing each value in your table with the graph and updating your table with correct values, verify the sorting is correct, and before proceeding, confirm all corrections have been made and use ONLY the corrected data for analysis. \n \n' +
                   'Task 4: Question Analysis: Using ONLY the verified data in your table, compare EACH value individually with the reference value, for "less than" comparisons mark ALL values that are even slightly below the reference, for "greater than" comparisons mark ALL values that are even slightly above the reference, and show each comparison on a new line. \n \n' +
                   'Provide your reasoning with specific references to table values. \n\n' +
                   'End with: "Correct Answer: ". Just write the value, nothing else. Do not write anything after this. \n\n' +
                   'Let\'s solve this step by step.'
      },
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": question_text
          },
          {
            "type": "image",
            "source": {
              "type": "base64",
              "media_type": "image/png",
              "data": base64_image
            }
          }
        ]
      }
    ]
  }
  
  # Add retry mechanism
  max_retries = 3
  retry_delay = 10  # seconds
  retry_count = 0
  
  while retry_count < max_retries:
    time_start = time.perf_counter()
    try:
      response_raw = requests.post(CLAUDE_API_URL, headers=headers, json=payload)
      time_end = time.perf_counter()
      response = response_raw.json()
      
      # Check if we got an overloaded error
      if 'error' in response and response.get('error', {}).get('type') == 'overloaded_error':
        retry_count += 1
        print(f"Iteration {i}: Got overloaded error. Retry {retry_count}/{max_retries} after {retry_delay} seconds...")
        time.sleep(retry_delay)
        # Increase the delay for subsequent retries
        retry_delay *= 2
        continue
      else:
        # Success or different error, break the retry loop
        break
        
    except Exception as e:
      time_end = time.perf_counter()
      response = {"error": {"message": str(e)}}
      break
  
  if 'error' in response:
    print(f"Iteration {i} error:", response, "\n")
    claude_answer = response["error"]["message"]
    extracted_answer = ""
    is_correct = False
  else:
    claude_answer = response["content"][0]["text"]
    
    print(f"Processing question {i}, answer in CSV: '{question_answer}'")
    
    # Extract the clean response (text after "Correct Answer: ")
    extracted_answer = extract_answer(claude_answer)
    
    # Compare with the correct answer (with 5% tolerance for numeric values)
    is_correct = False
    if extracted_answer:
        is_correct = compare_answers(extracted_answer, question_answer)
    
    print(f"Extracted answer: '{extracted_answer}', Correct answer: '{question_answer}', Match: {is_correct}")
    
  responses_data.append([claude_answer, time_end-time_start, extracted_answer, is_correct])
  
  # Print completion message for each iteration
  print(f"Iteration {i} done")
  
  if i % 10 == 0:
    print("Finished with question", i, "\n")
  
  # Save intermediate results every 20 iterations
  if i % 20 == 0:
    interim_df = pd.DataFrame(responses_data, columns=['response', 'time', 'response_clean', 'correct_bool'])
    interim_df.index = range(1, interim_df.shape[0] + 1)
    timestamp = str(int(time.time()))
    interim_filename = f"Claude_VisQA_Interim_{timestamp}_iterations_1-{i}.csv"
    interim_df.to_csv(interim_filename, index_label="id")
    print(f"Saved interim results up to iteration {i} at {interim_filename}")
    
  # Always wait 5 seconds between iterations regardless of response time
  print(f"Waiting 5 seconds before next iteration...")
  time.sleep(5)

results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'response_clean', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
timestamp = str(int(time.time()))
final_csv = f"Claude_VisQA_Complete_{timestamp}.csv"
results_df.to_csv(final_csv, index_label="id")
print(f"*** Finished ***")
print(f"Results saved to CSV: {final_csv}")